# 09 · RLHF: how a reward number becomes a weight update

**The question this notebook answers:** a reward model outputs a single number. Weights are updated by backpropagation. Backpropagation needs a differentiable path from the loss to the weights. **So where is that path?** The reward came from a different, frozen model, scoring text that was *sampled*, and sampling is not differentiable.

That confusion is correct and important. Most explanations skip straight to PPO's clipping and never answer it.

The short answer, which the rest of this notebook makes concrete:

> **The reward never receives a gradient. It acts as a multiplier.**
> You never backpropagate *through* the reward. You build a different loss, called a surrogate,
> whose gradient happens to equal the gradient you actually want. The reward enters that surrogate
> as a plain scalar coefficient on `log π(y|x)`, and `log π` **is** differentiable with respect to
> the weights. Backpropagation runs through `log π` alone.

By the end you will have derived that by hand, verified it against autograd, and watched weights move.

In [ ]:
# Connect this notebook to the lab's saved paths, data, scorer, and configuration.
# sys controls Python's import search path; json reads/writes JSON; warnings controls warning messages.
# Putting the project root on sys.path lets `from src import lab` find our own lab module.
# The relative '..' path assumes a suitable working directory; importing lab does not load an LLM.
# CFG is a nested dictionary: CFG['model']['name'] looks inside 'model', then reads 'name'.
# filterwarnings('ignore') hides warnings, not exceptions; useful loader advice may therefore be hidden.

import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])


## First, the decision: should you use RLHF at all?

Be clear about this before learning the machinery, because the machinery is the expensive part.

| | Needs a reward model | Needs generation during training | Models in memory | When it is right |
|---|---|---|---|---|
| **SFT** | no | no | 1 | Teach a format or skill you can demonstrate |
| **DPO** (notebook 06) | **no** | **no** | 2, one frozen | You have preference pairs. Almost always start here |
| **PPO / RLHF** | **yes** | **yes** | 4 | Large preference datasets, you need online exploration, and you can afford the instability |
| **GRPO / RLVR** | no, uses a verifier | yes | 2 | The answer is checkable, as in maths or code |

For a grounded product assistant, **DPO is the correct choice** and you already built it in notebook 06.

So why learn RLHF? Two reasons, both practical. It is what every frontier lab actually ran to make instruction-following models, so it is assumed knowledge in interviews. And DPO only makes sense once you understand what it replaced.

**A note on the tooling:** TRL 1.14, the version installed here, has **removed `PPOTrainer`**. It ships `RewardTrainer`, `GRPOTrainer` and `RLOOTrainer`. That is a real signal about where the field went. It also suits us: the PPO loop is written by hand below, which is where the learning is anyway.

---

# Part 1 · The mechanism, on three numbers

Before any model, shrink the problem until you can do the arithmetic in your head.

A "language model" with a vocabulary of **three tokens**: `good`, `okay`, `bad`. It has three logits. That is the entire model.

In [ ]:
# Build a three-action policy small enough to inspect every number.
# torch provides tensors and automatic differentiation; torch.nn.functional, named F, provides softmax.
# Here the three logits themselves are trainable values; there is no transformer yet.
# requires_grad=True tells PyTorch to record operations so a later backward() can differentiate them.

import torch
import torch.nn.functional as F

VOCAB = ["good", "okay", "bad"]

# The policy's raw scores. These ARE the weights for now.
logits = torch.tensor([0.5, 0.2, 0.3], requires_grad=True)
# softmax exponentiates and normalizes the scores so probabilities sum to one.
# dim=-1 selects the final dimension, which is the three possible tokens in this one-dimensional tensor.
probs = F.softmax(logits, dim=-1)

# zip pairs token labels with probabilities. .item() extracts one scalar as a Python number
# for display; it disconnects that extracted number from the gradient graph.
for t, p in zip(VOCAB, probs):
    print(f"  P({t:5s}) = {p.item():.4f}")
print(f"\nsum = {probs.sum().item():.4f}")


### One step of the loop, by hand

1. **Sample** an action from the policy. Say it produced `good`, index 0.
2. A **frozen reward model** scores it. Say it returns `R = +1.0`.
3. Now: update the weights so `good` becomes more likely.

Step 1 is the problem. `torch.multinomial` returns an integer index. There is no derivative of "which token got picked" with respect to the logits. The computational graph is severed right there.

So we do not differentiate the sampling. We use the **score function estimator**, also called REINFORCE:

$$\nabla_\theta \\; \mathbb{E}_{y \sim \pi_\theta}[R(y)] \\;=\\; \mathbb{E}_{y \sim \pi_\theta}\big[\, R(y) \, \nabla_\theta \log \pi_\theta(y) \,\big]$$

Read the right-hand side carefully. **The gradient is only ever taken of `log π`.** `R` sits outside the gradient operator as a scalar.

So to make an optimiser do the right thing, define a loss whose gradient equals that expression:

$$\mathcal{L} = -\,R \cdot \log \pi_\theta(y)$$

Minimising this loss maximises reward-weighted log-probability. That is the entire bridge from a reward number to a weight update.

In [ ]:
# Turn one chosen action and its scalar reward into a differentiable surrogate loss.
# The token index and reward are treated as fixed observations; no gradient passes through sampling or scoring.
# The gradient comes from the log-probability that the policy assigned to the chosen token.
# The leading minus sign lets ordinary loss minimization implement reward maximization.

action = 0          # we sampled "good"
R = 1.0             # the frozen reward model returned this

# Index 0 selects 'good'. log(p) is negative for 0<p<1; increasing p makes -R*log(p)
# smaller when R is positive, encouraging this sampled action.
log_prob = torch.log(probs[action])
loss = -(R * log_prob)

# These prints convert tensors with .item() only for inspection. The original loss tensor
# still retains the graph needed by the next backward() cell.
print(f"sampled       : {VOCAB[action]}")
print(f"P(sampled)    : {probs[action].item():.4f}")
print(f"log P(sampled): {log_prob.item():.4f}")
print(f"reward R      : {R}")
print(f"surrogate loss: {loss.item():.4f}   =  -({R} * {log_prob.item():.4f})")


### Now the part that answers the question: compute the gradient by hand

For a softmax policy, the derivative of `log π(a)` with respect to logit `i` has a famously clean form:

$$\frac{\partial \log \pi(a)}{\partial z_i} = \mathbb{1}[i = a] - \pi(i)$$

In words: **one minus its own probability for the token you sampled, and minus the probability for every other token.**

Our loss is `-R · log π(a)`, so:

$$\frac{\partial \mathcal{L}}{\partial z_i} = -R\,\big(\mathbb{1}[i=a] - \pi(i)\big)$$

Compute that with a pen, then let autograd compute it, and compare.

In [ ]:
# Check the policy-gradient derivative by hand against PyTorch autograd.
# For logit z_i, d[-R log p(action)]/dz_i = -R * (1[i=action] - p_i).
# The indicator is one for the sampled token and zero for the others; softmax couples all logits.
# This cell computes gradients only; it has no optimizer step and does not change the logits.

# by hand
# torch.empty allocates three slots without initializing them; the loop fills every slot.
# The conditional expression builds the indicator for each vocabulary position.
manual = torch.empty(3)
for i in range(3):
    indicator = 1.0 if i == action else 0.0
    manual[i] = -R * (indicator - probs[i].item())

# by autograd
# backward traverses the recorded computation graph and accumulates derivatives in .grad.
# Rerunning backward on the same freed graph is not the same as recomputing a fresh loss.
loss.backward()
auto = logits.grad

print(f"{'token':6s} {'prob':>8s} {'manual grad':>13s} {'autograd':>11s}")
for i, t in enumerate(VOCAB):
    print(f"{t:6s} {probs[i].item():8.4f} {manual[i].item():13.6f} {auto[i].item():11.6f}")

# allclose checks numerical agreement within a tolerance; floating-point calculations
# need not match bit for bit. atol=1e-6 is an absolute tolerance of one millionth.
print("\nidentical:", torch.allclose(manual, auto, atol=1e-6))


**That match is the answer to your question.**

The reward `R = 1.0` appears in the gradient as a plain multiplier. Nothing differentiated it. Nothing needed to. It scaled a gradient that came entirely from `log π`, which is an ordinary differentiable function of the weights.

Read the signs in the table:

- The sampled token `good` has a **negative** gradient. Gradient descent subtracts the gradient, so its logit **goes up**, and it becomes more likely.
- The other two have **positive** gradients, so their logits go down. Probability is conserved.

Now flip the reward and watch the whole thing reverse.

In [ ]:
# Isolate how reward sign and magnitude scale the same policy-gradient direction.
# Each function call makes fresh trainable logits and a fresh graph, avoiding accumulated old gradients.
# The reward is a fixed Python number multiplying a differentiable log-probability.
# A negative reward reverses the update direction; a zero reward gives zero gradient for this surrogate.

def grad_for(R_value, action=0):
    z = torch.tensor([0.5, 0.2, 0.3], requires_grad=True)
    p = F.softmax(z, dim=-1)
    (-(R_value * torch.log(p[action]))).backward()
    # clone copies the gradient values so the returned tensor remains a separate result.
    # No optimizer runs here: we inspect the derivative, not a trained new policy.
    return z.grad.clone()

print("effect of the reward's sign and size on the gradient\n")
# The generator expression makes one formatted header per vocabulary item; join combines them.
# :>9s right-aligns text, while :9.5f uses a width of nine and five decimal places.
print(f"{'R':>6s}  " + "  ".join(f"{t:>9s}" for t in VOCAB))
for R_value in [2.0, 1.0, 0.0, -1.0, -2.0]:
    g = grad_for(R_value)
    print(f"{R_value:6.1f}  " + "  ".join(f"{x:9.5f}" for x in g))

print("\nR =  0  -> zero gradient. No reward, no learning signal.")
print("R = -1  -> signs flip. The sampled token is pushed DOWN.")
print("R =  2  -> same direction as R=1, exactly twice the magnitude.")
print("\nThe reward is a scalar volume knob on a gradient it never touches.")


## Watch the weights actually move

Run the loop. Reward `good` with +1, `okay` with 0, `bad` with -1, and let it learn.

In [ ]:
# Run 300 tiny REINFORCE updates so the gradient mechanism has a visible consequence.
# manual_seed makes this sampling experiment reproducible; SGD updates the single logit tensor z.
# REWARD_TABLE is a fixed rule, not a learned reward model. No LLM or API is involved.

torch.manual_seed(0)
z = torch.tensor([0.5, 0.2, 0.3], requires_grad=True)
opt = torch.optim.SGD([z], lr=0.5)
REWARD_TABLE = {0: 1.0, 1: 0.0, 2: -1.0}

history = []
for step in range(300):
    p = F.softmax(z, dim=-1)
    # multinomial draws a categorical action from the probabilities; .item() makes its ID a Python integer.
    # This discrete sample is not differentiated. Its selected log-probability below is differentiated.
    a = torch.multinomial(p, 1).item()      # sample. NOT differentiable
    R = REWARD_TABLE[a]                      # frozen scorer. NOT differentiable
    loss = -(R * torch.log(p[a]))            # surrogate. IS differentiable
    # zero_grad clears old accumulated derivatives; backward calculates this sample's derivatives;
    # step subtracts learning_rate times the gradient from the logits. The semicolons separate those operations.
    opt.zero_grad(); loss.backward(); opt.step()
    # detach removes gradient history and clone saves a separate snapshot, avoiding a growing graph.
    # The first stored snapshot is already after one update; the later printed 'start' is not the original policy.
    history.append(F.softmax(z, dim=-1).detach().clone())

final = F.softmax(z, dim=-1)
print("start -> end")
for i, t in enumerate(VOCAB):
    print(f"  P({t:5s}) 0.{int(history[0][i]*10000):04d} -> {final[i].item():.4f}")


In [ ]:
# Plot the probability trajectory from the completed toy training loop.
# matplotlib.pyplot, called plt, builds charts; torch.stack turns the list into a [steps, 3] tensor.
# H[:, i] selects all steps for token i, so each curve follows one action probability.
# The axes label probability versus optimizer step; no new training occurs in this plotting cell.

import matplotlib.pyplot as plt
H = torch.stack(history)
fig, ax = plt.subplots(figsize=(8, 3.5))
for i, t in enumerate(VOCAB):
    ax.plot(H[:, i], label=f"P({t})  R={REWARD_TABLE[i]:+.0f}")
ax.set_xlabel("REINFORCE step"); ax.set_ylabel("probability")
ax.set_title("Reward shapes the policy, through log-prob gradients only")
# legend names the curves, grid adds reference lines, and tight_layout adjusts spacing.
# show renders the finished figure; alpha=.3 makes the grid partially transparent.
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()


That is RLHF in miniature. Everything that follows is variance reduction and safety rails bolted onto this one idea.

---

# Part 2 · Why raw REINFORCE is not usable, and what PPO adds

The loop above works on three tokens. On a real model it would be hopeless. Three problems, three fixes.

### Problem 1: variance

The gradient estimate depends on which token happened to be sampled. With a large vocabulary and long sequences, the estimate swings wildly and training is noise-dominated.

**Fix: subtract a baseline.** Replace reward `R` with **advantage** `A = R - b`, where `b` estimates the expected reward for this prompt. This is provably unbiased, because the baseline does not depend on the sampled action, and it cuts variance sharply.

The intuition: "was this response better *than usual for this prompt*", not "was this response good in absolute terms". Measure it.

In [ ]:
# Compare repeated one-sample gradient estimates with and without a reward baseline.
# numpy, named np, handles arrays and summary statistics; the policy is reset for every trial.
# Subtracting an action-independent baseline can reduce gradient noise without changing the expected gradient.
# Finite trial averages still differ; this is a sampling demonstration, not a proof from equal printed numbers.

import numpy as np
torch.manual_seed(0)

def estimate_gradient_variance(use_baseline, n_trials=400):
    grads = []
    for _ in range(n_trials):
        z = torch.tensor([0.5, 0.2, 0.3], requires_grad=True)
        p = F.softmax(z, dim=-1)
        a = torch.multinomial(p, 1).item()
        R = REWARD_TABLE[a]
        # a fixed baseline: the expected reward under the current policy
        # This baseline is the policy's expected reward: sum of probability times reward for each action.
        # .item() makes it a fixed scalar for this update rather than a differentiable path through the baseline.
        b = sum(p[i].item() * REWARD_TABLE[i] for i in range(3)) if use_baseline else 0.0
        A = R - b
        (-(A * torch.log(p[a]))).backward()
        # Copy each three-component gradient and convert the CPU tensor to a NumPy array.
        # The resulting array has shape [n_trials, 3], one noisy gradient vector per trial.
        grads.append(z.grad.clone().numpy())
    return np.array(grads)

raw = estimate_gradient_variance(False)
base = estimate_gradient_variance(True)
print(f"gradient std-dev, averaged over the 3 logits")
print(f"  raw reward       : {raw.std(axis=0).mean():.4f}")
print(f"  minus a baseline : {base.std(axis=0).mean():.4f}")
# The existing formula compares standard deviations, not variances, despite its printed label.
# axis=0 computes spread across trials for each logit; mean() then averages the three spreads.
print(f"  variance cut     : {100*(1 - base.std(axis=0).mean()/raw.std(axis=0).mean()):.1f}%")
print(f"\nmean gradient is unchanged (the estimator stays unbiased):")
print(f"  raw  {raw.mean(axis=0).round(4)}")
print(f"  base {base.mean(axis=0).round(4)}")


In real PPO that baseline is not a constant. It is produced by a **value model**, a second network with a scalar head that predicts expected reward from the prompt. That is the third model in memory, after the policy and the reward model. GRPO's main idea, later in this notebook, is to delete it.

### Problem 2: one sample, one update is wasteful

Generating text is the slowest part of the loop. You want several gradient steps from each batch of generations. But after the first step the policy has changed, so the samples were drawn from an **old** policy. Using them naively is statistically wrong.

**Fix: importance sampling.** Reweight by the ratio of new to old probability:

$$r_t(\theta) = \frac{\pi_\theta(y_t)}{\pi_{\theta_{old}}(y_t)}$$

### Problem 3: the ratio explodes

If the new policy assigns far higher probability than the old one, the ratio blows up and a single batch can destroy the model.

**Fix: clip it.** This is the whole of PPO:

$$\mathcal{L}^{PPO} = -\min\Big( r_t A_t,\\; \text{clip}(r_t, 1-\epsilon, 1+\epsilon)\, A_t \Big)$$

Work out what that `min` actually does, because it is asymmetric and the asymmetry is the point.

In [ ]:
# Work through PPO's clipped objective using scalar arithmetic before using tensors.
# ratio means current probability / probability under the policy that collected the sample.
# A is advantage: better-than-baseline samples have positive A; worse ones have negative A.
# PPO maximizes the smaller of unclipped and clipped objectives; a trainer minimizes its negative.

def ppo_objective(ratio, A, eps=0.2):
    unclipped = ratio * A
    # min(max(ratio, 1-eps), 1+eps) clamps the ratio into [0.8, 1.2] when eps=0.2.
    # Clipping is applied before multiplying by A, so negative advantages reverse the relevant inequality.
    clipped = min(max(ratio, 1-eps), 1+eps) * A
    return min(unclipped, clipped)

# For A>0, increasing the ratio above 1.2 earns no further improvement in this objective.
# This does not impose an absolute hard bound on policy change across a full optimization step.
print("advantage POSITIVE (+1): the response was better than expected\n")
print(f"{'ratio':>7s} {'unclipped':>11s} {'clipped':>9s} {'objective':>11s}  effect")
for r in [0.5, 0.8, 1.0, 1.2, 1.5, 3.0]:
    u, c = r*1.0, min(max(r,0.8),1.2)*1.0
    o = ppo_objective(r, 1.0)
    note = "capped, no runaway" if r > 1.2 else ""
    print(f"{r:7.2f} {u:11.2f} {c:9.2f} {o:11.2f}  {note}")

# For A<0, pushing an already reduced ratio below 0.8 earns no additional benefit.
# But making a bad action more likely (ratio>1.2) remains penalized by the unclipped branch.
print("\n\nadvantage NEGATIVE (-1): the response was worse than expected\n")
print(f"{'ratio':>7s} {'unclipped':>11s} {'clipped':>9s} {'objective':>11s}  effect")
for r in [0.5, 0.8, 1.0, 1.2, 1.5, 3.0]:
    u, c = r*-1.0, min(max(r,0.8),1.2)*-1.0
    o = ppo_objective(r, -1.0)
    note = "NOT capped, keeps pushing down" if r > 1.2 else ""
    print(f"{r:7.2f} {u:11.2f} {c:9.2f} {o:11.2f}  {note}")


Read those two tables together.

When the advantage is **positive**, the objective is capped once the ratio passes `1 + ε`. The update cannot run away on a single lucky sample.

When the advantage is **negative**, it is *not* capped in the same direction. If the policy has drifted to make a bad response much more likely, PPO keeps pushing it down hard. The clipping is deliberately one-sided.

### The fourth piece: the KL penalty

Even with clipping, a policy optimising a reward model will find its blind spots. This is **reward hacking**: output that scores highly and is useless. So RLHF adds a penalty for drifting from the frozen reference model:

$$R_{final} = R_{\theta}(x,y) - \beta \cdot \text{KL}\big(\pi_\theta \,\|\, \pi_{ref}\big)$$

If you recognise that `β` from notebook 06, you should. **DPO's `beta` is the same leash**, folded algebraically into the loss instead of applied as a separate penalty term.

---

# Part 3 · The reward model

So far the reward came from a lookup table. In real RLHF it comes from a trained model, and the way it is trained is worth knowing because **it never sees a score**. It only ever sees comparisons.

### The Bradley-Terry loss, by hand

Humans are bad at absolute scores and good at comparisons. So the data is preference pairs: for a prompt, which of two responses is better. The reward model learns a scalar `r(x,y)` such that preferred responses score higher:

$$\mathcal{L}_{RM} = -\log \sigma\big(r(x, y_{chosen}) - r(x, y_{rejected})\big)$$

Only the **difference** matters. The absolute scale is arbitrary, which is exactly why PPO needs a baseline to make the numbers meaningful.

In [ ]:
# Calculate the pairwise ranking loss used to train a scalar reward model.
# math provides exp and log. Bradley-Terry assigns P(chosen wins)=sigmoid(r_chosen-r_rejected).
# Minimizing its negative log increases the preferred answer's score relative to the rejected answer.
# Only their difference matters: adding the same constant to both scores leaves the loss unchanged.

import math
def bt_loss(r_chosen, r_rejected):
    return -math.log(1/(1+math.exp(-(r_chosen - r_rejected))))

print(f"{'r_chosen':>9s} {'r_rejected':>11s} {'margin':>8s} {'loss':>8s}  reading")
for rc, rr in [(3.0,1.0), (1.0,0.5), (1.0,1.0), (0.5,1.0), (1.0,3.0)]:
    m = rc - rr
    # This nested conditional expression turns the score margin into a short interpretation.
    # The 'probability' concerns a preference comparison, not factual accuracy of either response.
    note = ("confident and right" if m>1 else "right, unsure" if m>0
            else "no opinion" if m==0 else "WRONG")
    print(f"{rc:9.1f} {rr:11.1f} {m:8.1f} {bt_loss(rc,rr):8.4f}  {note}")
print("\nmargin 0 gives loss -log(0.5) = 0.6931, the no-information point.")


### Build preference data

Reuse the Meridian pairs from notebook 06 if you ran it, otherwise construct them here from the same facts. The pattern is the one that matters commercially: **chosen** cites the source or declines, **rejected** is fluent but invents a number.

In [ ]:
# Prepare preferred/rejected examples for reward-model training.
# random supplies a seeded generator; json loads saved pairs and prints an example readably.
# If notebook 06 already saved training pairs, reuse them; otherwise construct synthetic pairs from facts.
# Reading this cell does not train a model, and these labels are rule-made rather than human judgments.

import random, json
rng = random.Random(0)
gen = lab.GEN / "dpo_train.json"

if gen.exists():
    pairs = json.loads(gen.read_text())
    print(f"reusing {len(pairs)} pairs from notebook 06")
else:
    facts = lab.facts()["facts"]
    # This fallback corrupts numbers by multiplying and rounding. Zero remains zero,
    # and rounding floats to integers can alter precision; inspect pairs rather than assuming every rejection differs.
    def corrupt(v):
        return round(v*rng.choice([1.35,0.72,1.5])) if isinstance(v,(int,float)) else "Ivory and Jet Black"
    pairs = []
    for f in facts:
        u = f" {f['unit']}" if f["unit"] else ""
        pairs.append({"prompt": f"Could you tell me the {f['label']} for the {f['model']}?",
                      "chosen": f"The {f['model']} has a {f['label']} of {f['value']}{u}.",
                      "rejected": f"The {f['model']} has a {f['label']} of {corrupt(f['value'])}{u}."})
    # Unanswerable prompts come from the training-only refusal seed, never from evals/.
    # Each gets the same fabricated percentage answer, so the synthetic contrast is deliberately simplistic.
    seed = json.loads((lab.GEN / "sft_session" / "policy_refusal_seed.json").read_text())
    for q in [fam["questions"][0] for fam in seed if fam["kind"] == "refusal"][:9]:
        pairs.append({"prompt": q,
                      "chosen": "I don't have that in Meridian's documents. I can connect you with a dealer.",
                      "rejected": "It is approximately 62 percent of the original value."})
    rng.shuffle(pairs)
    print(f"built {len(pairs)} pairs from facts.json")

# Print one complete pair so you can inspect its prompt, chosen text, and rejected text before training.
print("\nexample:")
print(json.dumps(pairs[0], indent=2))


### Optional: let Gemini be the labeller (this is RLAIF, not RLHF)

RLHF means *human* feedback. Collecting it is slow and expensive, so a common substitute is **RLAIF**, where a stronger model provides the preference labels. Worth doing once so you can speak to the difference, and worth being precise about it in an interview: the method is identical, only the label source changes, and the ceiling becomes the labeller's judgement rather than a human's.

Needs `GEMINI_API_KEY` in your environment or `.env` (see `docs/SETUP.md`). Skip this cell if you would rather not spend the calls.

In [ ]:
# Optionally ask Gemini to choose between 12 synthetic answer pairs.
# When RUN_RLAIF is True, the branch creates an API client and makes external (billed) model calls.
# lab.gemini_client reads the existing API key at execution; the code does not print the key.
# This measures agreement with our synthetic labels, not an independently verified accuracy score.

RUN_RLAIF = lab.has_secret("GEMINI_API_KEY")   # runs only when a Gemini key is set (see docs/SETUP.md)

if RUN_RLAIF:
    client = lab.gemini_client()
    GEM = CFG["generation"]["gemini_model"]
    agree = 0; checked = 0
    # [:12] caps the sample. The triple-quoted f-string inserts the question and both candidate answers.
    # The source documents themselves are not supplied to this judge prompt, so it cannot verify all company facts.
    for p in pairs[:12]:
        prompt = f'''A customer asked: "{p["prompt"]}"

Response A: {p["chosen"]}
Response B: {p["rejected"]}

The assistant must only state facts from Meridian's official documents and must decline
when it does not have the information. Which response is better? Reply with exactly A or B.'''
        # try/except stops on the first call failure. temperature=0 reduces sampling randomness
        # but does not guarantee identical hosted responses. strip/upper normalize response formatting.
        try:
            out = client.models.generate_content(model=GEM, contents=prompt,
                    config={"temperature": 0}).text.strip().upper()
            checked += 1
            # startswith('A') counts any reply beginning with A; it is not strict response-schema validation.
            # The result is an agreement diagnostic on these pairs, not proof that either label source is correct.
            if out.startswith("A"): agree += 1
        except Exception as e:
            print("  call failed:", type(e).__name__); break
    if checked:
        print(f"Gemini agreed with our labels on {agree}/{checked} pairs ({100*agree/checked:.0f}%)")
        print("\nThat agreement rate IS the quality of your AI labeller. Below about 80% and")
        print("the reward model you train on these labels inherits the disagreement as noise.")


### Train the reward model

A small sequence-classification head on the same base model, with one output: the scalar reward. TRL's `RewardTrainer` implements the Bradley-Terry loss you hand-computed above.

In [ ]:
# Create a reward model that maps a prompt-plus-response to one scalar score.
# transformers loads a sequence-classification variant and tokenizer; torch provides tensors and MPS.
# PEFT's LoraConfig describes adapter training; TRL's RewardTrainer/RewardConfig implement pairwise ranking;
# datasets.Dataset supplies the trainer's row format. This cell allocates a second model, not a language-model answer.

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig
from trl import RewardTrainer, RewardConfig
from datasets import Dataset
import torch

MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# num_labels=1 requests a one-number classification head. Loading language-model weights
# into this architecture adds a task-specific scoring head, which needs reward training.
rm = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32).to(DEVICE)
# Tell the model which token is padding so its sequence-level score can use the appropriate
# non-padding position. rm.score is the learned linear head that produces the scalar.
rm.config.pad_token_id = tok.pad_token_id
print("reward model head:", rm.score)
print("output dimension :", rm.score.out_features, " <- one scalar, this is the reward")


In [ ]:
# Format reward pairs and configure LoRA reward-model training.
# Each row contains complete chosen/rejected texts including the same user prompt.
# The slice split reserves approximately 15% of pairs; the current trainer does not receive rm_eval.

def to_rm_rows(rows):
    return Dataset.from_list([{
        "chosen":   f"User: {r['prompt']}\nAssistant: {r['chosen']}",
        "rejected": f"User: {r['prompt']}\nAssistant: {r['rejected']}"} for r in rows])

split = int(len(pairs)*0.85)
rm_train, rm_eval = to_rm_rows(pairs[:split]), to_rm_rows(pairs[split:])

lc = CFG["training"]["lora"]
# Train for one pass with small batches and eight-step gradient accumulation.
# max_length=256 limits each text; eval_strategy='no' means no validation pass is scheduled.
rm_args = RewardConfig(output_dir=str(lab.RUNS/"reward_model"), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=1e-4,
    logging_steps=5, eval_strategy="no", save_strategy="no", report_to=[], max_length=256)
# SEQ_CLS marks a sequence-classification task so PEFT can handle its scoring head appropriately.
# The trainer uses differences between chosen/rejected scores to learn the pairwise preference loss.
rm_trainer = RewardTrainer(model=rm, args=rm_args, train_dataset=rm_train,
    processing_class=tok,
    peft_config=LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                           target_modules=lc["target_modules"], task_type="SEQ_CLS"))


In [ ]:
# Start reward-model training using the dataset and configuration prepared above.
# This is the expensive weight-updating step; inspect the pair example and token limit first.
# The returned metrics report what happened during optimization.

# This call runs optimization and changes reward-model trainable weights; initialization alone did not.
# The printed metrics describe training, not an independent quality benchmark.
res = rm_trainer.train()
print(res.metrics)


Watch `rewards/accuracies` in the log. That is the fraction of held-out pairs the reward model ranks correctly. **It is the ceiling on everything downstream.** A reward model at 60% accuracy cannot produce a good policy, no matter how well PPO is tuned, because the signal it emits is close to noise.

In [ ]:
# Check whether the trained reward model ranks a few reserved pairs in the right order.
# eval() disables training-mode behavior such as dropout; no_grad separately avoids a gradient graph.
# The helper returns a Python float, which intentionally disconnects reward scoring from policy gradients.

rm.eval()
def score(prompt, response):
    text = f"User: {prompt}\nAssistant: {response}"
    # The prompt and response are combined as text and converted to model inputs.
    # truncation=True can cut content beyond 256 tokens; this is the existing scoring setup.
    ids = tok(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        # **ids passes the tokenizer's named tensors to the model. logits has [batch, labels] shape;
        # [0][0] selects the first sequence's one scalar, and item() converts it to a Python number.
        return rm(**ids).logits[0][0].item()

print(f"{'prompt':<44s} {'chosen':>8s} {'rejected':>9s} {'margin':>8s}  ok?")
correct = 0
# The slice uses at most eight reserved pairs. sc>sr tests ranking rather than an absolute score scale.
# The displayed denominator is fixed at eight in this code, even if a different dataset yielded fewer pairs.
for p in pairs[split:split+8]:
    sc, sr = score(p["prompt"], p["chosen"]), score(p["prompt"], p["rejected"])
    ok = sc > sr; correct += ok
    print(f"{p['prompt'][:42]:<44s} {sc:8.3f} {sr:9.3f} {sc-sr:8.3f}  {'yes' if ok else 'NO'}")
print(f"\nranked correctly: {correct}/8")
print("\nNote the scores are not calibrated to any scale. Only differences carry meaning.")


---

# Part 4 · The full loop, written out on the real model

Now assemble it: sample from the policy, score with the frozen reward model, compute an advantage, build the surrogate loss, backpropagate.

**Deliberately written by hand rather than handed to a trainer**, because the whole point is to see where the gradient comes from. This is a teaching implementation, not a production one: one sample per prompt, a running-mean baseline instead of a value model, and no clipping until the cell after.

In [ ]:
# Load the policy that will generate responses and learn from the frozen reward scores.
# AutoModelForCausalLM predicts next tokens; PEFT's get_peft_model attaches new LoRA matrices.
# Only q_proj and v_proj receive rank-8 adapters here, a smaller setup than the earlier full-target adapters.
# This loads another model; it does not reuse the reward model's classification head.

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

policy = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
policy = get_peft_model(policy, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0,
        target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"))
policy.print_trainable_parameters()
# AdamW maintains optimizer state and updates only parameters selected by requires_grad.
# The list comprehension filters out frozen base weights; lr=1e-5 is the policy learning rate.
opt = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=1e-5)


In [ ]:
# Separate discrete sampling from the differentiable probability calculation.
# sample_response produces token IDs without a gradient graph; sequence_logprob recomputes their probabilities.
# That second forward pass is where reward-weighted learning can reach the policy's trainable weights.

def sample_response(prompt, max_new_tokens=40):
    """Generate, and return the tokens plus the prompt length.

    torch.no_grad() here is deliberate and is the crux of the whole notebook:
    the SAMPLING carries no gradient. We recompute log-probs with gradient
    afterwards, on the tokens that were actually produced.
    """
    text = tok.apply_chat_template(
        [{"role":"system","content":"You are the Meridian Motors assistant."},
         {"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(DEVICE)
    # Generate tokens without recording gradients. top_p=0.95 restricts sampling to a probability nucleus;
    # temperature=1 leaves score scale unchanged. This sampler differs from the full softmax used below.
    with torch.no_grad():
        out = policy.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=1.0, top_p=0.95, pad_token_id=tok.eos_token_id)
    # Select the first generated sequence and return its prompt length as a separate number.
    # The token sequence includes prompt plus response, so callers can identify the generated suffix.
    return out[0], ids["input_ids"].shape[1]


In [ ]:
# Recompute the sampled response's log-probability with autograd enabled.
# This helper does not sample new tokens; it scores the existing token sequence so gradients can flow.
# Follow the batch/time/vocabulary shapes below to see exactly which answer tokens contribute.



def sequence_logprob(full_ids, prompt_len):
    """Recompute log P(response | prompt) WITH gradient. This is the differentiable part."""
    # unsqueeze(0) adds a batch axis: [T] becomes [1,T]. No token values are changed.
    full = full_ids.unsqueeze(0)
    # The logits at position t predict token t+1. Remove the last prediction and the first target
    # so logits [1,T-1,V] align with target IDs [1,T-1]. V is vocabulary size.
    logits = policy(full).logits[:, :-1, :]          # predict token t+1 from t
    targets = full[:, 1:]
    # log_softmax normalizes scores over the vocabulary and takes logs stably in one operation.
    # dim=-1 means the vocabulary dimension, not time or batch.
    logp = torch.log_softmax(logits, dim=-1)
    # targets.unsqueeze(-1) has [1,T-1,1] shape. gather(2, ...) selects each actual target token's
    # log-probability from the vocabulary axis; squeeze(-1) removes that length-one axis again.
    # This is indexing, not averaging: one chosen-token log-probability remains at every time position.
    token_logp = logp.gather(2, targets.unsqueeze(-1)).squeeze(-1)
    # The first response token is predicted at prompt_len-1, so the slice starts one earlier than its ID.
    # Summing token log-probabilities gives log P(response|prompt), since log turns a probability product into a sum.
    # The prompt predictions are excluded; this one-sequence example has no padding mask to apply.
    return token_logp[0, prompt_len-1:].sum()        # response tokens only

print("helpers ready")


In [ ]:
# Run a small reward-weighted policy-gradient demonstration on generated responses.
# statistics is imported for the later variability summary. prompts selects up to sixteen question-like prompts.
# Each iteration samples text, gets a fixed reward, then differentiates the policy's log-probability of that text.
# This is a teaching loop, not the full PPO algorithm or a reliable quality experiment.

import statistics
prompts = [p["prompt"] for p in pairs[:24] if "?" in p["prompt"]][:16]
baseline, hist = None, []

for step, prompt in enumerate(prompts):
    full_ids, plen = sample_response(prompt)
    response = tok.decode(full_ids[plen:], skip_special_tokens=True).strip()

    R = score(prompt, response)                      # frozen RM, no gradient
    # An exponential moving average uses 90% of the old baseline and 10% of the current reward.
    # The first baseline equals its reward, so its first advantage and gradient are zero.
    # Because this baseline includes the current sample, it is not the action-independent baseline from the earlier proof.
    baseline = R if baseline is None else 0.9*baseline + 0.1*R
    A = R - baseline                                 # advantage

    logp = sequence_logprob(full_ids, plen)          # differentiable
    # A is a scalar: multiplying by logp rescales the whole response gradient.
    # A>0 increases the sampled response's likelihood; A<0 decreases it, all else equal.
    loss = -(A * logp)                               # the surrogate

    opt.zero_grad(); loss.backward()
    # clip_grad_norm_ rescales gradients together if their total norm exceeds 1.0.
    # The trailing underscore denotes an in-place operation; its return value is the norm before clipping.
    gnorm = torch.nn.utils.clip_grad_norm_(
        [p for p in policy.parameters() if p.requires_grad], 1.0)
    opt.step()

    # Store Python scalars rather than computation graphs. step % 4 == 0 prints every fourth iteration;
    # : +7.3f-style formatting exposes reward signs and three decimal places in aligned columns.
    hist.append({"R": R, "baseline": baseline, "A": A,
                 "logp": logp.item(), "loss": loss.item(), "grad": gnorm.item()})
    if step % 4 == 0:
        print(f"[{step:2d}] R={R:+7.3f} base={baseline:+7.3f} A={A:+7.3f} "
              f"logP={logp.item():8.2f} loss={loss.item():9.2f} |g|={gnorm:.3f}")
        print(f"      {response[:90]}")


### Read that log carefully, because it is the whole answer

For each row:

- `R` came from the reward model. **No gradient. It is a Python float.**
- `A = R - baseline`. Still just a float.
- `logP` is `log π(response | prompt)` recomputed **with** gradient, through the policy weights.
- `loss = -(A × logP)`. The float multiplies the differentiable tensor.
- `.backward()` computes `∂loss/∂weights = -A × ∂logP/∂weights`.

**The reward scaled a gradient. It never had one.** That is the mechanism, and it is identical to the three-token example, only with 494 million parameters and a 40-token response instead of one.

Notice also: when `A` is positive the loss pushes that response's log-probability up; when negative, down. Same sign logic as the toy.

In [ ]:
# Inspect reward, advantage, and gradient size from the completed demonstration loop.
# plt.subplots(1,3) creates a row of three axes; each axis receives its own history values.
# The list comprehensions extract one named measurement from every saved step; no further training occurs.

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot([h["R"] for h in hist], label="reward")
axes[0].plot([h["baseline"] for h in hist], label="baseline", ls="--")
axes[0].set_title("reward vs baseline"); axes[0].legend(); axes[0].grid(alpha=.3)
# The horizontal zero line separates positive and negative advantages.
# A bar below zero means this sampled response was pushed down relative to the baseline.
axes[1].axhline(0, color="k", lw=.8)
axes[1].bar(range(len(hist)), [h["A"] for h in hist])
axes[1].set_title("advantage = R - baseline"); axes[1].grid(alpha=.3)
axes[2].plot([h["grad"] for h in hist])
axes[2].set_title("gradient norm"); axes[2].grid(alpha=.3)
for a in axes: a.set_xlabel("step")
plt.tight_layout(); plt.show()

# statistics.pstdev computes population standard deviation for the recorded advantages.
# Sixteen changing prompts and noisy samples illustrate mechanics, not a stable estimate of training progress.
print(f"advantage std-dev: {statistics.pstdev([h['A'] for h in hist]):.3f}")
print("On 16 steps with one sample each, this is pure noise. Real PPO runs thousands")
print("of steps with large batches and a learned value model. The MECHANISM is what")
print("this cell demonstrates; the learning is not.")


### Add the PPO pieces to the same loop

Now the ratio, the clip and the KL penalty, on top of the loop you just ran. This is the honest minimal PPO.

In [ ]:
# Demonstrate a PPO-like update that reuses one sampled response for several inner steps.
# EPS sets ratio clipping width; BETA weights a reference-policy log-ratio penalty.
# The code uses one sequence-level advantage and ratio, not full tokenwise PPO with a learned value function.
# Read the frozen-old-policy and frozen-reference paths carefully: they play different roles.

EPS, BETA = 0.2, 0.05

def ppo_step(prompt, ref_logp=None, n_inner=2):
    full_ids, plen = sample_response(prompt)
    response = tok.decode(full_ids[plen:], skip_special_tokens=True).strip()
    R = score(prompt, response)

    # Freeze the current policy's log-probability at sample collection time.
    # .detach() explicitly disconnects it so later losses cannot update their own denominator.
    with torch.no_grad():                            # the OLD policy's log-prob
        old_logp = sequence_logprob(full_ids, plen).detach()
    with torch.no_grad():                            # the frozen REFERENCE
        # This nested context manager temporarily disables LoRA, exposing the original base as reference.
        # Leaving the with block restores the adapter; it does not overwrite the adapter weights.
        with policy.disable_adapter():
            ref = sequence_logprob(full_ids, plen).detach()

    # old_logp-ref is a signed log-likelihood ratio for this one sampled response.
    # Its expectation under the sampling policy relates to KL; this individual value is not an exact nonnegative KL.
    kl = (old_logp - ref).item()                     # a sample estimate of KL
    R_total = R - BETA * kl                          # KL-penalised reward
    # The existing `baseline or R_total` uses Python truthiness: a baseline of zero chooses R_total,
    # just as None does. The optional ref_logp function argument is not used by this implementation.
    A = R_total - (baseline or R_total)

    logs = []
    for _ in range(n_inner):                         # reuse the sample, hence the ratio
        new_logp = sequence_logprob(full_ids, plen)
        # exp(new_logp-old_logp) equals P_new(response)/P_old(response), using logs to form the ratio.
        # The old value remains fixed while the new value is recomputed with gradients each inner step.
        ratio = torch.exp(new_logp - old_logp)
        unclipped = ratio * A
        # clamp restricts the ratio to [1-EPS,1+EPS]; the scalar bounds broadcast over this tensor.
        # min chooses the conservative objective; the leading minus sign converts maximization into loss minimization.
        clipped = torch.clamp(ratio, 1-EPS, 1+EPS) * A
        loss = -torch.min(unclipped, clipped)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in policy.parameters() if p.requires_grad], 1.0)
        opt.step()
        logs.append({"ratio": ratio.item(), "loss": loss.item()})
    return {"R": R, "kl": kl, "R_total": R_total, "A": A, "inner": logs,
            "response": response[:70]}


In [ ]:
# Execute the PPO-like helper on one prompt, then inspect its reward and ratio arithmetic.
# This cell really updates the policy; defining ppo_step above did not run it.
# The stored short response is for inspection and is not the complete token-level training record.


# Calling the helper now samples a response and performs optimizer updates.
# The printout exposes the reward penalty and inner-step ratios; it is not a production PPO training report.
out = ppo_step(prompts[0])
print(f"reward        {out['R']:+.3f}")
print(f"KL from ref   {out['kl']:+.3f}")
print(f"penalised R   {out['R_total']:+.3f}   = R - {BETA} * KL")
print(f"advantage     {out['A']:+.3f}")
for i, l in enumerate(out["inner"]):
    print(f"  inner {i}: ratio={l['ratio']:.4f}  loss={l['loss']:+.4f}")
print(f"\nresponse: {out['response']}")


The first inner step always has `ratio` exactly 1.0, because the new policy and the old policy are still the same model. After the optimiser moves, the second inner step's ratio drifts away from 1. **That drift is precisely what the clip bounds.**

---

# Part 5 · GRPO, and why PPO is going out of fashion

PPO needs four models in memory: policy, reference, reward model and value model. On a 70B model that is brutal.

**GRPO's idea is to delete the value model.** Instead of learning a baseline, sample **several** responses to the same prompt and use the group's mean reward as the baseline. The advantage becomes:

$$A_i = \frac{R_i - \text{mean}(R_{group})}{\text{std}(R_{group})}$$

You already know this is legitimate: any baseline that does not depend on the sampled action is unbiased, and the group mean qualifies. Compute it by hand.

In [ ]:
# Show how group-relative advantages can be calculated without a learned value model.
# mean finds the group center; pstdev finds its population standard deviation.
# Each normalized advantage is (reward - group_mean)/group_std, a unitless relative score.
# The example has nonzero spread; an all-equal reward group would need a zero-division guard.

group_rewards = [2.1, -0.4, 0.8, 1.6, -1.2, 0.3]
m = statistics.mean(group_rewards); s = statistics.pstdev(group_rewards)
print(f"group rewards: {group_rewards}")
print(f"mean {m:.3f}  std {s:.3f}\n")
print(f"{'R':>7s} {'R - mean':>10s} {'normalised A':>14s}")
# Positive means above the group mean, negative means below. Centering makes the advantages sum
# approximately to zero; it does not guarantee exactly half positive and half negative in every group.
for R in group_rewards:
    print(f"{R:7.2f} {R-m:10.3f} {(R-m)/s:14.3f}")
print("\nHalf the group gets a positive advantage, half negative, by construction.")
print("No value model was trained. That is the entire saving.")


In [ ]:
# Configure a GRPO trainer using a deliberately simple Python reward function.
# TRL's GRPOTrainer generates groups of responses and forms relative advantages; GRPOConfig holds settings.
# The rule below rewards refusal phrases and penalizes digits. It does not actually verify grounding in documents.
# This design can reward over-refusal or penalize correct numerical answers, so it is a mechanism example.

from trl import GRPOTrainer, GRPOConfig

# completions contains generated answers. **kwargs accepts additional named inputs supplied by TRL,
# so this callback can ignore arguments it does not need without raising an unexpected-keyword error.
def reward_fn(completions, **kwargs):
    """GRPO accepts a plain Python function as the reward. It can be the trained
    reward model, a verifier, or a rule. Here: reward grounded refusals and
    penalise invented numbers."""
    out = []
    for c in completions:
        # The conditional expression accepts either a plain string or a chat-message representation.
        # any(...) returns True once a refusal phrase or digit is found; each completion gets one numeric reward.
        text = c if isinstance(c, str) else c[0]["content"]
        r = 0.0
        if any(k in text.lower() for k in ("don't have", "do not have", "not in the")):
            r += 1.0
        if any(ch.isdigit() for ch in text):
            r -= 0.5
        out.append(r)
    return out


In [ ]:
# Create the prompt dataset and GRPO configuration, then construct the trainer.
# The reward_fn above is passed as a callable, not executed manually here.
# The next cell's train() call performs the generation-and-update loop.


# Dataset.from_list turns eight prompt dictionaries into the trainer's input table.
# num_generations=4 means each prompt gets a group of four sampled completions.
grpo_ds = Dataset.from_list([{"prompt": p} for p in prompts[:8]])
# beta weights the reference penalty; learning_rate controls adapter updates.
# max_completion_length caps generated text, while batch size controls prompts processed in a training batch.
gcfg = GRPOConfig(output_dir=str(lab.RUNS/"grpo"), num_generations=4, beta=0.04,
    num_train_epochs=1, per_device_train_batch_size=4, learning_rate=1e-5,
    max_completion_length=40, logging_steps=1, report_to=[], save_strategy="no")
gtrainer = GRPOTrainer(model=policy, reward_funcs=reward_fn, args=gcfg,
                       train_dataset=grpo_ds, processing_class=tok)
print("GRPO trainer ready. The next cell trains; it is slow because it generates 4x per prompt.")


In [ ]:
# Start GRPO training with the policy, prompts, and reward rule configured above.
# Each training batch generates multiple responses, scores the group, and updates trainable weights.
# This can be slow because generation is part of training; it is not just a fixed-text forward pass.
# gres.metrics summarizes the training run, not a separate factual-quality evaluation.

gres = gtrainer.train()
print(gres.metrics)


---

## Where this leaves DPO

Now the comparison means something, because you have run both.

| | RLHF with PPO | GRPO | DPO (notebook 06) |
|---|---|---|---|
| Reward model | trained separately | a function or verifier | **none** |
| Value model | yes | **none** | none |
| Generation during training | yes, the slow part | yes, several per prompt | **no** |
| Models in memory | 4 | 2 | 2, one frozen |
| Where the gradient comes from | `A × ∇log π`, the surrogate | same, group baseline | closed-form preference loss |
| Stability | notoriously fiddly | better | well-behaved |
| Needs | preference data plus compute | a checkable reward | **just preference pairs** |

DPO's contribution was showing that for the specific reward model used in RLHF, the optimal policy has a closed form. Rearrange it and the reward model cancels out, leaving a supervised-looking classification loss over preference pairs. **No sampling, no reward model, no value model, no clipping.** That is why it took over for preference alignment, and why `beta` in notebook 06 is the same KL leash you applied by hand here.

GRPO survives where DPO cannot reach: when the answer is *checkable*, as in maths or code, a verifier gives a real online reward signal that static preference pairs cannot.

## What you should be able to say now

1. **Where the gradient comes from.** The reward is a scalar multiplier on `∇log π(y|x)`. Nothing differentiates the reward, and sampling is never differentiated. You recompute log-probs with gradient on tokens that were already produced.
2. **Why a baseline exists.** Variance. You measured the reduction yourself.
3. **What PPO's clip does**, and that it is deliberately one-sided.
4. **What `beta` controls**, and that it is the same quantity in DPO.
5. **Why the reward model's accuracy is the ceiling** on everything downstream.
6. **Why you would choose DPO** for a grounded product assistant, and when GRPO beats it.

One honest framing for interviews: this notebook demonstrates the mechanism on sixteen steps and one sample per prompt. It does not demonstrate that the policy improved. Saying that distinction out loud is worth more than claiming you ran RLHF.